# Apache Impala

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Apache Impala is a massively parallel (MPP) SQL query engine for data stored in HDFS, object storage, and Apache Kudu. It runs as long-lived daemons (`impalad`) on every DataNode, avoids MapReduce entirely, and returns interactive results over Hive-compatible tables.

The questions below mix classic Impala interview questions with modern operational detail: DDL, partitioning, statistics, join planning, Parquet vs Kudu, resource pools, EXPLAIN plans, and real production scenarios.

## Contents

<details>
<summary>40 sections - click to expand</summary>

- [How do I try Impala out?](#how-do-i-try-impala-out)
- [How much Memory is required?](#how-much-memory-is-required)
- [What are the main features of Impala?](#what-are-the-main-features-of-impala)
- [What features from relational databases or hive are not available in Impala?](#what-features-from-relational-databases-or-hive-are-not-available-in-impala)
- [Are results returned as they become available or all at once when a query completes?](#are-results-returned-as-they-become-available-or-all-at-once-when-a-query-completes)
- [Why does my select statement fail?](#why-does-my-select-statement-fail)
- [Does Impala performance improve as it is deployed to more hosts in a cluster in much the same way that Hadoop performance does?](#does-impala-performance-improve-as-it-is-deployed-to-more-hosts-in-a-cluster-in-much-the-same-way-that-hadoop-performance-does)
- [Is the Hdfs Block Size reduced to achieve faster query results?](#is-the-hdfs-block-size-reduced-to-achieve-faster-query-results)
- [Does Impala use caching?](#does-impala-use-caching)
- [What are the good use cases for Impala as opposed to Hive or MapReduce?](#what-are-the-good-use-cases-for-impala-as-opposed-to-hive-or-mapreduce)
- [Is Impala intended to handle Real Time Queries in Low latency Applications or is it for ad Hoc Queries for the purpose of data exploration?](#is-impala-intended-to-handle-real-time-queries-in-low-latency-applications-or-is-it-for-ad-hoc-queries-for-the-purpose-of-data-exploration)
- [Is Hive an Impala requirement?](#is-hive-an-impala-requirement)
- [What happens if there is an Error in Impala?](#what-happens-if-there-is-an-error-in-impala)
- [How are Joins performed in Impala?](#how-are-joins-performed-in-impala)
- [What is Impala's aggregation strategy?](#what-is-impalas-aggregation-strategy)
- [How is Impala Metadata managed?](#how-is-impala-metadata-managed)
- [How does Impala achieve its performance improvements?](#how-does-impala-achieve-its-performance-improvements)
- [What happens when the data set exceeds available memory?](#what-happens-when-the-data-set-exceeds-available-memory)
- [What are the most Memory intensive operations?](#what-are-the-most-memory-intensive-operations)
- [Is there an update statement?](#is-there-an-update-statement)
- [Can Impala do User defined functions?](#can-impala-do-user-defined-functions)
- [Why do I have to use refresh and invalidate Metadata and what do they do?](#why-do-i-have-to-use-refresh-and-invalidate-metadata-and-what-do-they-do)
- [Why is space not freed up when I issue drop table?](#why-is-space-not-freed-up-when-i-issue-drop-table)
- [How do I load a big Csv file into a partitioned table?](#how-do-i-load-a-big-csv-file-into-a-partitioned-table)
- [Can I Do Insert Select Into a partitioned table?](#can-i-do-insert-select-into-a-partitioned-table)
- [How do you optimize Impala queries?](#how-do-you-optimize-impala-queries)
- [How do you create Impala tables with DDL and choose a file format?](#how-do-you-create-impala-tables-with-ddl-and-choose-a-file-format)
- [How do you create partitioned tables and manage partitions in Impala?](#how-do-you-create-partitioned-tables-and-manage-partitions-in-impala)
- [How do COMPUTE STATS and INVALIDATE METADATA drive Impala query plans?](#how-do-compute-stats-and-invalidate-metadata-drive-impala-query-plans)
- [How do you read an Impala EXPLAIN plan and tune joins and hints?](#how-do-you-read-an-impala-explain-plan-and-tune-joins-and-hints)
- [How do you use Parquet and Kudu tables in Impala?](#how-do-you-use-parquet-and-kudu-tables-in-impala)
- [How do Impala resource pools and admission control work?](#how-do-impala-resource-pools-and-admission-control-work)
- [How do you handle small files and table maintenance in Impala?](#how-do-you-handle-small-files-and-table-maintenance-in-impala)
- [What are Impala's trade-offs and when should you not use it?](#what-are-impalas-trade-offs-and-when-should-you-not-use-it)
- [Tuning 1: statistics and metadata for good plans](#tuning-1-statistics-and-metadata-for-good-plans)
- [Things to remember: key numbers and limits](#things-to-remember-key-numbers-and-limits)
- [Things to remember: semantics to memorize](#things-to-remember-semantics-to-memorize)
- [Things to remember: mistakes that keep coming back](#things-to-remember-mistakes-that-keep-coming-back)
- [Real world use cases for Apache Impala](#real-world-use-cases-for-apache-impala)
- [Impala exercises](#impala-exercises)

</details>

Tip: the links open the sections on GitHub; in Jupyter or Colab, use the notebook's outline panel in the sidebar instead.

## How do I try Impala out?

To look at the core features and functionality on Impala, the easiest way to try out Impala is to download the Cloudera QuickStart VM and start the Impala service through Cloudera Manager, then use `impala-shell` in a terminal window or the Impala Query UI in the Hue web interface.

To do performance testing and try out the management features for Impala on a cluster, you need to move beyond the QuickStart VM with its virtualized single-node environment. Ideally, download the Cloudera Manager software to set up the cluster, then install the Impala software through Cloudera Manager.

```bash
# Interactive shell against a coordinator
impala-shell -i coordinator-host:21000 -d analytics

# Run a query file and exit
impala-shell -i coordinator-host:21000 -f /home/etl/load_orders.sql

# Non-interactive one-liner
impala-shell -i coordinator-host:21000 -q "select count(*) from analytics.fact_orders"
```

**Mental model.** An Impala cluster is three long-lived services on top of Hadoop. `impalad` runs on every worker, accepts client SQL, plans it, and executes query fragments; whichever daemon takes your connection coordinates that query. `statestored` tracks the live `impalad` membership and pushes small state updates to them. `catalogd` owns the catalog: it loads table metadata from the Hive metastore, runs DDL, and broadcasts catalog changes to every `impalad`.

**How a query flows.** Client to coordinator `impalad` (port `21050` for HiveServer2/JDBC, `21000` for the beeswax shell protocol). The planner asks `catalogd` for table metadata and the NameNode for block locations, then dispatches plan fragments to `impalad` workers. Each fragment scans local HDFS blocks, results are exchanged between fragments, and the coordinator assembles and streams rows back. Per-daemon web UIs expose live state: `impalad` on `25000`, `statestored` on `25010`, `catalogd` on `25020`.

**Worked example and common mistake.**

```bash
# Point the shell at any impalad; that node coordinates this session
impala-shell -i coordinator-host:21000 -d analytics

# One-shot query with delimited output
impala-shell -i coordinator-host:21000 -B --output_delimiter=',' \
  -q "select count(*) from analytics.fact_orders"

# Run a script and keep a full log of statements and timings
impala-shell -i coordinator-host:21000 -f etl/refresh_orders.sql -o /tmp/refresh.log
```

Common mistake: treating `-i` as "the master node" to install or pin to. Every `impalad` can coordinate, so clients should connect through a load balancer or any healthy daemon, not one hard-coded host.

## How much Memory is required?

Although Impala is not an in-memory database, when dealing with large tables and large result sets, you should expect to dedicate a substantial portion of physical memory for the `impalad` daemon. Recommended physical memory for an Impala node is 128 GB or higher. If practical, devote approximately 80% of physical memory to Impala.

The amount of memory required for an Impala operation depends on several factors:

+ The file format of the table. Different file formats represent the same data in more or fewer data files. The compression and encoding for each file format might require a different amount of temporary memory to decompress the data for analysis.
+ Whether the operation is a SELECT or an INSERT. For example, Parquet tables require relatively little memory to query, because Impala reads and decompresses data in 8 MB chunks. Inserting into a Parquet table is a more memory-intensive operation because the data for each data file (potentially hundreds of megabytes, depending on the value of the `PARQUET_FILE_SIZE` query option) is stored in memory until encoded, compressed, and written to disk.
+ Whether the table is partitioned or not, and whether a query against a partitioned table can take advantage of partition pruning.
+ Whether the final result set is sorted by the ORDER BY clause. Each Impala node scans and filters a portion of the total data, and applies the LIMIT to its own portion of the result set. In Impala 1.4.0 and higher, if the sort operation requires more memory than is available on any particular host, Impala uses a temporary disk work area to perform the sort. The intermediate result sets are all sent back to the coordinator node, which does the final sorting and then applies the LIMIT clause to the final result set.

For example, if you execute the query:

```sql
select * from giant_table order by some_column limit 1000;
```

and your cluster has 50 nodes, then each of those 50 nodes will transmit a maximum of 1000 rows back to the coordinator node. The coordinator node needs enough memory to sort (LIMIT * cluster_size) rows, although in the end the final result set is at most LIMIT rows, 1000 in this case.

Likewise, if you execute the query:

```sql
select * from giant_table where test_val > 100 order by some_column;
```

then each node filters out a set of rows matching the WHERE conditions, sorts the results (with no size limit), and sends the sorted intermediate rows back to the coordinator node. The coordinator node might need substantial memory to sort the final result set, and so might use a temporary disk work area for that final phase of the query.

Whether the query contains any join clauses, GROUP BY clauses, analytic functions, or DISTINCT operators. These operations all require some in-memory work areas that vary depending on the volume and distribution of data. In Impala 2.0 and later, these kinds of operations utilize temporary disk work areas if memory usage grows too large to handle.

The size of the result set. When intermediate results are being passed around between nodes, the amount of data depends on the number of columns returned by the query. For example, it is more memory-efficient to query only the columns that are actually needed in the result set rather than always issuing `SELECT *`.

The mechanism by which work is divided for a join query. You use the COMPUTE STATS statement, and query hints in the most difficult cases, to help Impala pick the most efficient execution plan.

Operationally, memory is governed by `--mem_limit` at startup, the `MEM_LIMIT` query option (`set mem_limit = 32gb;`), and admission control through resource pools.

**Mental model.** Per-query memory is not a function of table size. It comes from a few operators: scan buffers and decompression, hash tables for join build sides and `GROUP BY`, sort and analytic buffers, exchange buffers, and result batches. `impalad` allocates through `tcmalloc` and keeps freed memory cached for reuse. Two caps govern it: `--mem_limit` at daemon startup and the `MEM_LIMIT` query option; admission control estimates a query's memory before it is allowed to run.

**Worked example.** A 256 GB worker keeps roughly 80 percent for Impala, so `--mem_limit=200gb`. A dashboard query that peaks near 8 GB and a nightly aggregation that peaks near 40 GB mix as roughly `200 / 8` interactive slots or `200 / 40` heavy slots, but not both at once. Pools and `MEM_LIMIT` enforce that split. Common mistake: sizing memory from input bytes. A `GROUP BY` costs memory proportional to the number of groups, and a hash join costs memory proportional to the build side after filtering.

```sql
-- Per-query cap, useful when a pool is shared
set mem_limit = 16gb;

-- See what a query actually reserved and whether it spilled
set explain_level = 1;
```

Common mistake: leaving `MEM_LIMIT` unset in a shared pool. One bad query can then reserve the whole node and push every other query into the admission-control queue.

## What are the main features of Impala?

A large set of SQL statements, including SELECT and INSERT, with joins, Subqueries in Impala SELECT Statements, and Impala Analytic Functions. Highly compatible with HiveQL, and also including some vendor extensions.

+ Distributed, high-performance queries.
+ Using Cloudera Manager, you can deploy and manage your Impala services. Cloudera Manager is the best way to get started with Impala on your cluster.
+ Using Hue for queries.
+ Appending and inserting data into tables through the INSERT statement.
+ ODBC: Impala is certified to run against MicroStrategy and Tableau, with restrictions. For more information, see Configuring Impala to Work with ODBC.
+ Querying data stored in HDFS and HBase in a single query.
+ In Impala 2.2.0 and higher, querying data stored in the Amazon Simple Storage Service (S3).
+ Concurrent client requests. Each Impala daemon can handle multiple concurrent client requests. The effects on performance depend on your particular hardware and workload.
+ Kerberos authentication.
+ Partitions. With Impala SQL, you can create partitioned tables with the CREATE TABLE statement, and add and drop partitions with the ALTER TABLE statement. Impala also takes advantage of the partitioning present in Hive tables.

**Mental model.** Impala's feature list follows from its architecture: MPP execution inside `impalad` with no MapReduce, LLVM runtime code generation, vectorized row batches, columnar Parquet and ORC scans with predicate pushdown, runtime filters built from join build sides, partitioned tables, complex types, Kudu upserts, and Iceberg reads. Security features cover Kerberos authentication, Sentry/Ranger authorization, and column masking.

**Gotcha.** Every item is version-dependent. `select version();` tells you what the cluster runs, and modern Apache Impala 4.x has features (Iceberg write support, `UPDATE`/`DELETE` on Iceberg, new resource management) that older CDH 5 answers miss. Answer interview questions at the level of the version you actually operate.

## What features from relational databases or hive are not available in Impala?

+ Querying streaming data.
+ Deleting individual rows. You delete data in bulk by overwriting an entire table or partition, or by dropping a table.
+ Indexing (not currently). LZO-compressed text files can be indexed outside of Impala, as described in Using LZO-Compressed Text Files.
+ Full text search on text fields. The Cloudera Search product is appropriate for this use case.
+ Custom Hive Serializer/Deserializer classes (SerDes). Impala supports a set of common native file formats that have built-in SerDes in CDH.
+ Checkpointing within a query. That is, Impala does not save intermediate results to disk during long-running queries. Currently, Impala cancels a running query if any host on which that query is executing fails. When one or more hosts are down, Impala reroutes future queries to only use the available hosts, and Impala detects when the hosts come back up and begins using them again. Because a query can be submitted through any Impala node, there is no single point of failure. In the future, we will consider adding additional work allocation features to Impala, so that a running query would complete even in the presence of host failures.
+ Encryption of data transmitted between Impala daemons.
+ Hive indexes.
+ Non-Hadoop data stores, such as relational databases.

Important modern correction: row-level `UPDATE` and `DELETE` do exist for Kudu-backed tables, and since Impala 4.3 also for Iceberg tables. There is still no row-level `UPDATE`/`DELETE` for plain HDFS/Parquet tables, where you change data by overwriting a partition or table; the classic "no row-level update" answer remains accurate for those.

```sql
-- Supported for Kudu tables: point update and delete
update analytics.customer_kudu set tier = 'gold' where customer_id = 1001;
delete from analytics.customer_kudu where customer_id = 1002;
```

**Mental model.** What is missing is driven by the storage layer, not by SQL taste. HDFS files are immutable and written in bulk, so plain Parquet tables have no row-level `UPDATE` or `DELETE`, no secondary indexes, no per-row constraints, and no multi-statement transactions. Row-level change needs a mutable store: Kudu tables with a primary key, or Iceberg tables in Impala 4.3 and later.

**Gotcha.** Kudu `UPDATE` and `DELETE` need equality predicates on every primary key column so the statement can target exact rows. A predicate on a non-key column forces a scan and can be rejected or extremely slow. On plain Parquet, the idempotent pattern is `INSERT OVERWRITE` of one partition, not a row update.

## Are results returned as they become available or all at once when a query completes?

Impala streams results whenever they are available, when possible. Certain SQL operations (aggregation or ORDER BY) require all of the input to be ready before Impala can return results.

Practical consequence: a `select ... limit 10` over a large table can paint the first rows quickly, while `group by` over the same table must finish its aggregation phase before returning anything. BI tools that show partial results therefore benefit from projection and filter pushdown rather than from LIMIT tricks.

**Mental model.** Operators are either streaming or blocking. `SCAN`, `FILTER`, and `HASH JOIN` probe sides stream rows as they arrive. `SORT`, `TOP-N` with a full ordering, `AGGREGATE` finalize, and `ANALYTIC` are blocking: they must consume all input before producing the first output row. The coordinator streams whatever the root operator emits.

**Gotcha.** `LIMIT` does not automatically make a blocking plan fast. If the plan contains a `SORT` or a final aggregation, the limit applies after that operator drains its input. Only a `TOP-N` operator or an early scan limit lets Impala stop reading sooner, so check the plan before promising instant first rows.

## Why does my select statement fail?

When a SELECT statement fails, the cause usually falls into one of the following categories:

+ A timeout because of a performance, capacity, or network issue affecting one particular node.
+ Excessive memory use for a join query, resulting in automatic cancellation of the query.
+ A low-level issue affecting how native code is generated on each node to handle particular WHERE clauses in the query. For example, a machine instruction could be generated that is not supported by the processor of a certain node. If the error message in the log suggests the cause was an illegal instruction, consider turning off native code generation temporarily, and trying the query again.
+ Malformed input data, such as a text data file with an enormously long line, or with a delimiter that does not match the character specified in the `FIELDS TERMINATED BY` clause of the CREATE TABLE statement.

Diagnose with the query profile rather than guessing:

**Mental model.** Most failures are resource or data problems, not SQL parser problems. Classify the message first: memory limit exceeded (build side too large, wrong join strategy), connection or timeout (one slow node, network), decode error (bad file, wrong delimiter, wrong `FIELDS TERMINATED BY`), or an illegal instruction (codegen assumption about the CPU).

**Worked example.** If the log mentions an illegal instruction, turn off native code generation for one query instead of restarting the daemon.

```sql
set disable_codegen = true;
-- then rerun the failing query; if it passes, the issue is codegen/CPU specific
```

Common mistake: rerunning blindly. Pull the query profile, find the operator with the highest `TotalTime` or the largest `RowsSpilled`, and fix that operator.

## Why does my insert statement fail?

When an INSERT statement fails, it is usually the result of exceeding some limit within a Hadoop component, typically HDFS.

An INSERT into a partitioned table can be a strenuous operation due to the possibility of opening many files and associated threads simultaneously in HDFS. Impala 1.1.1 includes some improvements to distribute the work more efficiently, so that the values for each partition are written by a single node, rather than as a separate data file from each node.

Certain expressions in the SELECT part of the INSERT statement can complicate the execution planning and result in an inefficient INSERT operation. Try to make the column data types of the source and destination tables match up, for example by doing `ALTER TABLE ... REPLACE COLUMNS` on the source table if necessary. Try to avoid CASE expressions in the SELECT portion, because they make the result values harder to predict than transferring a column unchanged or passing the column through a built-in function.

Be prepared to raise some limits in the HDFS configuration settings, either temporarily during the INSERT or permanently if you frequently run such INSERT statements as part of your ETL pipeline.

The resource usage of an INSERT statement can vary depending on the file format of the destination table. Inserting into a Parquet table is memory-intensive, because the data for each partition is buffered in memory until it reaches the configured file size, at which point the data file is written to disk. Impala can distribute the work for an INSERT more efficiently when statistics are available for the source table that is queried during the INSERT statement.

```sql
-- Control Parquet writer memory and output file size, then load a single year
set PARQUET_FILE_SIZE = 256m;
set NUM_SCANNER_THREADS = 8;

insert into analytics.fact_orders partition (year)
select order_id, customer_id, amount, year
from staging.orders_raw
where year = 2026;
```

**Mental model.** An `INSERT` into a partitioned table opens a writer per output partition per node. With dynamic partitioning and many partitions, the coordinator and the DataNodes run out of file handles, threads, or memory before HDFS runs out of space. Parquet worst case: each partition's data is buffered in memory until it reaches `PARQUET_FILE_SIZE`.

**Worked example and common mistake.**

```sql
set num_dynamic_partitions = 400;   -- cap partitions touched by one INSERT
set num_scanner_threads = 8;        -- cap concurrent scan threads per node

insert into analytics.fact_orders partition (order_date)
select order_id, customer_id, amount, cast(order_date as date)
from staging.orders_clean
where order_date between '2026-01-01' and '2026-01-31';
```

Common mistake: one `INSERT` that creates five years of daily partitions. Split it by month or year so each statement opens a bounded number of writers. Also avoid `CASE` expressions over the projected columns, which complicate type matching and distribute badly.

## Does Impala performance improve as it is deployed to more hosts in a cluster in much the same way that Hadoop performance does?

Yes. Impala scales with the number of hosts. It is important to install Impala on all the DataNodes in the cluster, because otherwise some nodes must do remote reads to retrieve data not available for local reads. Data locality is an important architectural aspect for Impala performance.

Scaling is close to linear for scan-heavy queries, but not for every query shape: a broadcast join must copy the small side to every node, so adding nodes increases fan-out. Watch the `DataStreamSender`/`Exchange` rows in `explain` before assuming more nodes will help.

**Mental model.** Scan throughput scales close to linearly with workers because each one reads its local blocks. Everything else scales worse: a broadcast join copies the build side to every worker (fan-out grows with the cluster), a shuffle join pays network proportional to the data, and the coordinator alone does final aggregation, sorting, and result assembly.

**Gotcha.** Adding nodes does not fix a coordinator-bound query or a small-file table. If the profile shows most time in a single `EXCHANGE` or in planning over hundreds of thousands of files, more workers make it no faster. Watch `DataStreamSender` and `EXCHANGE` rows before assuming scale-out is the answer.

## Is the Hdfs Block Size reduced to achieve faster query results?

No. Impala does not make any changes to the HDFS or HBase data sets.

The default Parquet block size is relatively large (256 MB in Impala 2.0 and later; 1 GB in earlier releases). You can control the block size when creating Parquet files using the `PARQUET_FILE_SIZE` query option. The underlying HDFS block size stays whatever the cluster is configured for (commonly 128 MB), and Parquet files are simply written as multi-block files.

**Mental model.** Three sizes are easy to confuse. The HDFS block size (commonly 128 MB) is how the file is spread across DataNodes. The Parquet file size is set when Impala writes, via `PARQUET_FILE_SIZE` (default 256 MB in Impala 2.0 and later). The Parquet row group inside that file is the unit whose statistics enable predicate pushdown. Impala never rewrites HDFS block settings.

**Gotcha.** Making HDFS blocks tiny does not speed up scans; it raises NameNode metadata and open costs. Control the Parquet file size instead.

```sql
set parquet_file_size = 256m;   -- Impala writes multi-block Parquet files
```

The underlying HDFS block size stays as the cluster configured it, and one Parquet file simply spans several HDFS blocks.

## Does Impala use caching?

Impala does not cache table data. It does cache some table and file metadata. Although queries might run faster on subsequent iterations because the data set was cached in the OS buffer cache, Impala does not explicitly control this.

Impala takes advantage of the HDFS caching feature in CDH. You can designate which tables or partitions are cached through the `CACHED` and `UNCACHED` clauses of the CREATE TABLE and ALTER TABLE statements. Impala can also take advantage of data that is pinned in the HDFS cache through the `hdfscacheadmin` command.

For object storage (S3/ADLS), OS cache pinning does not apply; use the storage layer's own cache or keep hot aggregates materialized in Kudu.

**Mental model.** There are three separate caches. The metadata cache holds table and file metadata inside `impalad` so planning does not hit the metastore every time. The OS page cache holds recently read blocks and makes repeat scans faster. The HDFS caching layer pins chosen data in memory on each DataNode, requested through `CACHED` and `UNCACHED` clauses on `CREATE TABLE` and `ALTER TABLE`.

**Gotcha.** Impala has no result cache. Re-running the same query returns the same rows by recomputing them, only faster if the OS or HDFS cache is warm. On object storage there is no OS pinning, so use the storage layer's cache or materialize hot aggregates into Kudu.

## What are the good use cases for Impala as opposed to Hive or MapReduce?

Impala is well-suited to executing SQL queries for interactive exploratory analytics on large data sets. Hive and MapReduce are appropriate for very long running, batch-oriented tasks such as ETL.

Rules of thumb:

+ Impala: dashboards, ad-hoc SQL, BI extracts over Parquet/Kudu, sub-second to tens of seconds.
+ Hive on Tez/Spark: heavy ETL, complex multi-stage transformations, very large joins that must spill, jobs measured in minutes to hours.
+ MapReduce: legacy, rarely the right answer for new work.

**Mental model.** Choose by latency budget, data mutability, and file format. Impala serves sub-second to tens-of-seconds SQL over Parquet, ORC, Kudu, and Iceberg. Hive on Tez or Spark does heavy multi-stage ETL where jobs run for minutes to hours. MapReduce is legacy for new work.

**Gotcha.** Do not run heavy ETL transformations in Impala just because the SQL looks convenient. Every intermediate `INSERT ... SELECT` writes a full copy to HDFS and re-runs statistics collection. Land curated data with Spark or Hive, then serve it from Impala.

## Is Impala intended to handle Real Time Queries in Low latency Applications or is it for ad Hoc Queries for the purpose of data exploration?

Ad-hoc queries are the primary use case for Impala. We anticipate it being used in many other situations where low-latency is required. Whether Impala is appropriate for any particular use-case depends on the workload, data size and query volume.

Kudu-backed tables extend Impala toward operational analytics: single-digit-millisecond key lookups and fast scans on the same table, which fits "real-time dashboards" better than pure HDFS Parquet.

**Mental model.** Impala's design point is interactive analytics: ad-hoc SQL, dashboards, and BI extracts over large columnar data. Kudu and Iceberg extend it toward operational analytics with key lookups and row-level change, which is what makes near-real-time dashboards possible.

**Gotcha.** Concurrency is the limit, not single-query speed. Hundreds of simultaneous queries need resource pools and memory caps; without them, latency degrades for everyone. Decide the SLA, then size pools from it.

## Is Hive an Impala requirement?

The Hive metastore service is a requirement. Impala shares the same metastore database as Hive, allowing Impala and Hive to access the same tables transparently.

Hive itself is optional, and does not need to be installed on the same nodes as Impala. Currently, Impala supports a wider variety of read (query) operations than write (insert) operations; you use Hive to insert data into tables that use certain file formats.

In practice, most clusters run both: Hive/Spark for the write-heavy ETL path and Impala for the read-heavy serving path, against the same metastore.

**Mental model.** The Hive metastore is the shared catalog service; Impala requires it. Hive itself (HiveServer2, the Tez or MapReduce execution engine) is optional and does not need to run on the same hosts. Many clusters run both against the same metastore: Spark or Hive writes, Impala reads.

**Gotcha.** People say "Hive is a requirement" and then debug the wrong process. The metastore is a standalone Thrift service (commonly port `9083`) with its own database; HiveServer2 is the query endpoint. Check the metastore when Impala cannot see tables, not HiveServer2.

## What happens if there is an Error in Impala?

There is not a single point of failure in Impala. All Impala daemons are fully able to handle incoming queries. If a machine fails however, all queries with fragments running on that machine will fail. Because queries are expected to return quickly, you can just rerun the query if there is a failure.

The longer answer: Impala must be able to connect to the Hive metastore. Impala aggressively caches metadata so the metastore host should have minimal load. Impala relies on the HDFS NameNode, and, in CDH4, you can configure HA for HDFS. Impala also has centralized services, known as the statestore and catalog services, that run on one host only. Impala continues to execute queries if the statestore host is down, but it will not get state updates. For example, if a host is added to the cluster while the statestore host is down, the existing instances of `impalad` running on the other hosts will not find out about this new host. Once the statestore process is restarted, all the information it serves is automatically reconstructed from all running Impala daemons.

If `catalogd` is down, DDL still works locally but metadata changes are not broadcast, so other coordinators may plan against stale metadata until it recovers. Monitor `catalog.ready` on the catalogd metrics endpoint.

**Mental model.** There is no single point of failure for running queries. Any `impalad` can coordinate, and fragments run on many workers. If a worker dies, every query with a fragment there fails, and because queries are short you normally just rerun. Statestore loss does not stop queries but freezes membership updates. Catalogd loss lets DDL run locally but stops broadcast of catalog changes to other coordinators.

**Gotcha.** Monitor `catalog.ready` on the catalogd metrics endpoint. If it drops, other coordinators plan against stale metadata and users see "table not found" for tables that exist. Restart recovery repopulates state automatically, but the window matters during ETL.

## How are Joins performed in Impala?

By default, Impala automatically determines the most efficient order in which to join tables using a cost-based method, based on their overall size and number of rows. (This is a new feature in Impala 1.2.2 and higher.) The COMPUTE STATS statement gathers information about each table that is crucial for efficient join performance. Impala chooses between two techniques for join queries, known as "broadcast joins" and "partitioned joins".

+ **Broadcast join (default for a small side):** the small table is sent to every node, and each node joins it with its local slice of the large table. Fast when the small side fits comfortably in memory; disastrous when it does not.
+ **Partitioned (shuffle) join:** both sides are hash-partitioned on the join key, so matching rows land on the same node. Scales to two large tables but pays network and disk cost.

```sql
-- Force a broadcast of the small dimension (only when it is genuinely small)
select /* +BROADCAST */ ...
from analytics.fact_orders o
join analytics.dim_region r on o.region_id = r.region_id;

-- Force a shuffle/partitioned join for two large tables
select /* +SHUFFLE */ ...
from analytics.fact_orders o
join analytics.fact_returns r on o.order_id = r.order_id;

-- Let the optimizer decide, but give it good statistics
compute stats analytics.fact_orders;
compute stats analytics.dim_region;
```

Interview point: hints override the planner, so use them to fix a *known* bad plan, not as a default. Runtime filters (Impala 2.5+) let a broadcast build side filter the probe side automatically.

**Mental model.** Impala plans a join as a tree of `HASH JOIN` operators and picks a distribution method per join. A broadcast join ships the small build side to every probe node, so each node joins it against its local slice and no data is shuffled. A partitioned (shuffle) join hash-partitions both sides on the join key through an `EXCHANGE`, so matching rows land on the same node. Broadcast wins when the build side fits in memory; shuffle is the only option when both sides are large.

**How the choice is made.** The cost-based optimizer uses statistics to estimate the cardinality of each input after filters. It prefers broadcast when the build side is small relative to the probe side, and it also reorders joins so the smallest selective join runs first. `COMPUTE STATS` on both sides is what makes this decision good. Runtime filters are then built from the build side and pushed down to the probe scan, which cuts probe-side rows before the exchange.

```sql
-- Force broadcast only when the build side is genuinely small
select /* +BROADCAST */ o.order_id, r.region
from analytics.fact_orders o
join analytics.dim_region r on o.region_id = r.region_id;

-- Force a shuffle for two large tables
select /* +SHUFFLE */ a.order_id, b.reason
from analytics.fact_orders a
join analytics.fact_returns b on a.order_id = b.order_id;

-- Give the optimizer real numbers instead of hints
compute stats analytics.fact_orders;
compute stats analytics.dim_region;
```

Common mistake: leaving stats stale and then "fixing" the bad plan with a permanent broadcast hint. Hints override the planner, so a hint that was right at one million rows becomes a memory failure at one hundred million.

## How does Impala process Join Queries for large tables?

Impala utilizes multiple strategies to allow joins between tables and result sets of various sizes. When joining a large table with a small one, the data from the small table is transmitted to each node for intermediate processing. When joining two large tables, the data from one of the tables is divided into pieces, and each node processes only selected pieces.

The plan is visible in `explain`:

`PARTITIONED` means a shuffle; `BROADCAST` means the right-hand side was replicated. If a plan shows `BROADCAST` on a table with hundreds of millions of rows, that is the bug to fix first.

**Mental model.** For two large tables the planner inserts an `EXCHANGE` that repartitions both inputs on the join key. Each node then builds one partition of the build side in memory and streams its probe partition through it. Memory is bounded by the largest build partition, not the whole table, which is why shuffle joins scale to tables far larger than node memory.

**Gotcha.** Shuffle joins are sensitive to key skew. If one join key holds a large share of rows, one node builds a huge hash table while others idle, and that node spills or fails. Detect it in the profile by comparing `RowsProduced` and `PeakMemoryUsage` across fragment instances; fix it by filtering, salting the key, or pre-aggregating the skewed side.

## What is Impala's aggregation strategy?

Impala currently only supports in-memory hash aggregation. In Impala 2.0 and higher, if the memory requirements for a join or aggregation operation exceed the memory limit for a particular host, Impala uses a temporary work area on disk to help the query complete successfully.

Two-phase aggregation reduces the size of the shuffle: partial aggregation happens where the scan runs, then final aggregation happens after the exchange.

Prefer `count(distinct ...)` on a low-cardinality column, and pre-aggregate high-cardinality GROUP BYs upstream in Spark/ETL when the grouping set is huge.

**Mental model.** Aggregation is hash-based and two-phase. A pre-aggregation runs where the scan happens to shrink the input, then an `EXCHANGE` repartitions by grouping key and a finalize `AGGREGATE` merges the partial states. This cuts shuffle volume dramatically. Memory is proportional to the number of distinct groups on a node, not to the number of input rows.

**Gotcha.** A `GROUP BY` on a near-unique column has almost no pre-aggregation benefit and a very large hash table. Use `ndv(col)` when an approximate distinct count is acceptable, and pre-aggregate high-cardinality groupings upstream in Spark or Hive. Watch `RowsSpilled` on the aggregate node as the signal that the group count exceeded memory.

## How is Impala Metadata managed?

Impala uses two pieces of metadata: the catalog information from the Hive metastore and the file metadata from the NameNode. Currently, this metadata is lazily populated and cached when an `impalad` needs it to plan a query.

The `REFRESH` statement updates the metadata for a particular table after loading new data through Hive. The `INVALIDATE METADATA` statement refreshes all metadata, so that Impala recognizes new tables or other DDL and DML changes performed through Hive.

In Impala 1.2 and higher, a dedicated `catalogd` daemon broadcasts metadata changes due to Impala DDL or DML statements to all nodes, reducing or eliminating the need to use the `REFRESH` and `INVALIDATE METADATA` statements.

**Mental model.** Two metadata layers live inside `impalad`. The catalog layer (tables, columns, partitions, statistics, file format) comes from the Hive metastore through `catalogd`. The file layer (which files exist, their block locations and sizes) comes from the HDFS NameNode. Both are loaded lazily on first use and cached, so planning does not hit the metastore on every query.

**Refresh versus invalidate.** `REFRESH` reloads the file/block metadata for known tables, which is what you need after an external writer adds files. `INVALIDATE METADATA` discards and reloads the catalog structure, which is what you need after external DDL such as `CREATE TABLE`, `ALTER TABLE`, or `ADD PARTITION`. Both are broadcast by `catalogd` to every `impalad`, so one node can issue them.

**Why DDL is slow.** Every catalog change is serialized through `catalogd`, written to the metastore, and broadcast to all `impalad` nodes; with `SYNC_DDL=true` the statement also waits for every node to acknowledge. `INVALIDATE METADATA` on a large table re-fetches every partition and file from the metastore and NameNode, which is why it should be scoped to one table and run off-peak. Common mistake: running `INVALIDATE METADATA` with no table name during business hours and stalling the whole cluster's planning.

## How does Impala achieve its performance improvements?

These are the main factors in the performance of Impala versus that of other Hadoop components and related technologies.

Impala avoids MapReduce. While MapReduce is a great general parallel processing model with many benefits, it is not designed to execute SQL. Impala avoids the inefficiencies of MapReduce in these ways:

+ Impala does not materialize intermediate results to disk. SQL queries often map to multiple MapReduce jobs with all intermediate data sets written to disk.
+ Impala avoids MapReduce start-up time. For interactive queries, the MapReduce start-up time becomes very noticeable. Impala runs as a service and essentially has no start-up time.
+ Impala can more naturally disperse query plans instead of having to fit them into a pipeline of map and reduce jobs. This enables Impala to parallelize multiple stages of a query and avoid overheads such as sort and shuffle when unnecessary.

Impala uses a more efficient execution engine by taking advantage of modern hardware and technologies:

+ Impala generates runtime code. Impala uses LLVM to generate assembly code for the query that is being run. Individual queries do not have to pay the overhead of running on a system that needs to be able to execute arbitrary queries.
+ Impala uses available hardware instructions when possible. Impala uses the supplemental SSE3 (SSSE3) instructions which can offer tremendous speedups in some cases. (Impala 2.0 and 2.1 required the SSE4.1 instruction set; Impala 2.2 and higher relax the restriction again so only SSSE3 is required.)
+ Impala uses better I/O scheduling. Impala is aware of the disk location of blocks and is able to schedule the order to process blocks to keep all disks busy.
+ Impala is designed for performance. A lot of time has been spent in designing Impala with sound performance-oriented fundamentals, such as tight inner loops, inlined function calls, minimal branching, better use of cache, and minimal memory usage.

Modern additions on top of the classic list: runtime filters, codegen for Parquet scanning with dictionary/filter pushdown, and the columnar memory layout shared with Kudu.

**Mental model.** Four mechanisms stack. Runtime code generation uses LLVM to compile the query's inner loops to native code with constants folded in. Vectorized execution processes rows in batches, which amortizes branching and keeps data in cache. The columnar layout reads only the columns a query needs and skips row groups using min/max statistics. I/O scheduling spreads scan ranges across disks to keep them all busy.

**Gotcha.** Codegen has a fixed cost. On very short queries over small tables, compilation can dominate, so Impala has `disable_codegen_rows_threshold` (default 5000) to skip codegen below that row count. Setting `disable_codegen=true` globally is a diagnostic step for CPU or illegal-instruction errors, not a performance setting.

## What happens when the data set exceeds available memory?

Currently, if the memory required to process intermediate results on a node exceeds the amount available to Impala on that node, the query is cancelled. You can adjust the memory available to Impala on each node, and you can fine-tune the join strategy to reduce the memory required for the biggest queries. We do plan on supporting external joins and sorting in the future.

Keep in mind though that the memory usage is not directly based on the input data set size. For aggregations, the memory usage is the number of rows after grouping. For joins, the memory usage is the combined size of the tables excluding the biggest table, and Impala can use join strategies that divide up large joined tables among the various nodes rather than transmitting the entire table to each node.

If a broadcast join repeatedly exceeds memory, force a shuffle join or pre-aggregate the build side.

**Mental model.** Operators that support spilling write their in-memory state to a scratch directory on local disk and continue. In modern Impala this covers hash joins, hash aggregation, sorting, and analytic functions. Memory is bounded by `MEM_LIMIT` and by admission control; disk scratch is bounded by `SCRATCH_LIMIT` and the `--scratch_dirs` configuration.

**Gotcha.** Not every operator can spill. Nested-loop joins and some broadcast join builds cannot, so they fail with `Memory limit exceeded` instead of degrading. If the profile shows spill on every node, adding memory helps; if the failure is on a broadcast build side, changing the strategy to `SHUFFLE` is the real fix.

## What are the most Memory intensive operations?

If a query fails with an error indicating "memory limit exceeded", you might suspect a memory leak. The problem could actually be a query that is structured in a way that causes Impala to allocate more memory than you expect, exceeded the memory allocated for Impala on a particular node. Some examples of query or table structures that are especially memory-intensive are:

+ INSERT statements using dynamic partitioning, into a table with many different partitions. (Particularly for tables using Parquet format, where the data for each partition is held in memory until it reaches the full block size in size before it is written to disk.) Consider breaking up such operations into several different INSERT statements, for example to load one year at a time rather than for all years at once.
+ GROUP BY on a unique or high-cardinality column. Impala allocates some handler structures for each different value in a GROUP BY query. Having millions of different GROUP BY values could exceed the memory limit.
+ Queries involving very wide tables, with thousands of columns, particularly with many STRING columns. Because Impala allows a STRING value to be up to 32 KB, the intermediate results during such queries could require substantial memory allocation.

**Mental model.** Rank the cost drivers by operator state, not by input size: dynamic-partition `INSERT` into a Parquet table (one in-memory writer per open partition), `GROUP BY` on a high-cardinality key (one entry per distinct group), hash join build sides, `ORDER BY` without a limit (full sort state), and very wide rows with many `STRING` columns (up to 32 KB per value).

**Gotcha.** Dynamic partition inserts are the classic production incident. Loading one year at once can open hundreds of Parquet writers per node; loading one month at a time keeps memory flat. Cap the blast radius with `num_dynamic_partitions` and split the load by time range.

## Is there an update statement?

Impala historically did not have an UPDATE statement, because HDFS files are optimized for bulk operations across many megabytes of data at a time, making traditional single-row UPDATE operations inefficient or impractical.

You can use the following techniques to achieve the same goals as the familiar UPDATE statement, in a way that preserves efficient file layouts for subsequent queries:

+ Replace the entire contents of a table or partition with updated data that you have already staged in a different location, either using `INSERT OVERWRITE`, `LOAD DATA`, or manual HDFS file operations followed by a `REFRESH` statement for the table. Optionally, you can use built-in functions and expressions in the INSERT statement to transform the copied data in the same way you would normally do in an UPDATE statement, for example to turn a mixed-case string into all uppercase or all lowercase.
+ To update a single row, use an HBase table, and issue an `INSERT ... VALUES` statement using the same key as the original row. Because HBase handles duplicate keys by only returning the latest row with a particular key value, the newly inserted row effectively hides the previous one.
+ To update rows at scale, use a Kudu table, which supports real `UPDATE` and `DELETE` with primary keys.

```sql
-- Bulk "update" by overwriting one partition
insert overwrite analytics.fact_orders partition (order_date = '2026-10-04')
select order_id, customer_id, upper(status) as status, amount
from staging.orders_fixed
where order_date = '2026-10-04';

-- True row-level update on a Kudu table
update analytics.customer_kudu
set tier = 'platinum', updated_at = now()
where customer_id = 1001;
```

**Mental model.** The answer depends on the storage format. Plain HDFS/Parquet tables are immutable files, so change is expressed as `INSERT OVERWRITE` of a partition or table. Kudu tables have primary keys and support real `UPDATE`, `DELETE`, and `UPSERT`. Iceberg tables support row-level `UPDATE` and `DELETE` in Impala 4.3 and later.

**Worked example and common mistake.**

```sql
-- Idempotent bulk change on immutable files: overwrite one partition
insert overwrite analytics.fact_orders partition (order_date = '2026-10-04')
select order_id, customer_id, upper(status), amount
from staging.orders_fixed
where order_date = '2026-10-04';

-- Row-level change on Kudu, targetable by primary key
update analytics.customer_kudu set tier = 'platinum' where customer_id = 1001;
```

Common mistake: using `INSERT INTO` for a correction. It appends a second copy of the rows. Use `INSERT OVERWRITE` scoped to the partition so reruns are safe.

## Can Impala do User defined functions?

Impala 1.2 and higher does support UDFs and UDAs. You can either write native Impala UDFs and UDAs in C++, or reuse UDFs (but not UDAs) originally written in Java for use with Hive.

Reusing Hive Java UDFs is common during migration because the jar already exists in the cluster. Keep UDFs small and side-effect free; they run in the query hot path on every node.

**Mental model.** Scalar UDFs and UDAs are native shared libraries loaded into every `impalad`; a Java Hive UDF can be reused for scalar work through its class name. UDFs run in the query hot path, inside the codegen'd pipeline, so they are compiled and called for every row. There is no UDTF support and no SerDes extension point.

**Gotcha.** A UDF that throws or returns nulls inconsistently can fail queries on one node and succeed elsewhere, which makes debugging hard. Keep UDFs small, deterministic, and side-effect free, and test them against the built-in equivalent before shipping.

## Why do I have to use refresh and invalidate Metadata and what do they do?

In Impala 1.2 and higher, there is much less need to use the `REFRESH` and `INVALIDATE METADATA` statements:

+ The new `impala-catalog` service, represented by the `catalogd` daemon, broadcasts the results of Impala DDL statements to all Impala nodes. Thus, if you do a CREATE TABLE statement in Impala while connected to one node, you do not need to do INVALIDATE METADATA before issuing queries through a different node.
+ The catalog service only recognizes changes made through Impala, so you must still issue a `REFRESH` statement if you load data through Hive or by manipulating files in HDFS, and you must issue an `INVALIDATE METADATA` statement if you create a table, alter a table, add or drop partitions, or do other DDL statements in Hive.
+ Because the catalog service broadcasts the results of `REFRESH` and `INVALIDATE METADATA` statements to all nodes, in the cases where you do still need to issue those statements, you can do that on a single node rather than on every node, and the changes will be automatically recognized across the cluster, making it more convenient to load balance by issuing queries through arbitrary Impala nodes rather than always using the same coordinator node.

The distinction to remember: `REFRESH` re-reads data/file metadata for known tables, while `INVALIDATE METADATA` re-reads catalog structure (tables, columns, partitions). Use the narrowest one that fixes the problem.

```sql
-- External writer added files to an existing partition
refresh analytics.fact_orders partition (order_date = '2026-10-04');

-- External writer created a brand new table
invalidate metadata analytics.new_table;
```

**Mental model.** The catalog service only knows about changes made through Impala, because Impala's own DDL is broadcast by `catalogd`. Any write that bypasses the catalog, such as Hive DDL, Spark writes, or direct HDFS file placement, leaves `impalad` caches stale. `REFRESH` re-reads file metadata for known tables; `INVALIDATE METADATA` re-reads catalog structure.

**Decision rule.** New files in an existing table or partition: `REFRESH`. New table, renamed column, added or dropped partition, changed format: `INVALIDATE METADATA`. Both are broadcast to all nodes, so issue on one coordinator. Scope to one table whenever possible.

```sql
-- External writer added files to an existing partition
refresh analytics.fact_orders partition (order_date = '2026-10-04');

-- Hive created a brand new table (or changed its columns)
invalidate metadata analytics.new_table;

-- Make DDL wait until all coordinators have the change
set sync_ddl = true;
```

Common mistake: running `INVALIDATE METADATA` cluster-wide instead of `REFRESH` on one table. The full invalidate re-reads everything from the metastore and NameNode and can stall planning for minutes.

## Why is space not freed up when I issue drop table?

Impala deletes data files when you issue a DROP TABLE on an internal table, but not an external one. By default, the CREATE TABLE statement creates internal tables, where the files are managed by Impala. An external table is created with a CREATE EXTERNAL TABLE statement, where the files reside in a location outside the control of Impala. Issue a `DESCRIBE FORMATTED` statement to check whether a table is internal or external. The keyword `MANAGED_TABLE` indicates an internal table, from which Impala can delete the data files. The keyword `EXTERNAL_TABLE` indicates an external table, where Impala will leave the data files untouched when you drop the table.

Even when you drop an internal table and the files are removed from their original location, you might not get the hard drive space back immediately. Deleted HDFS files normally go into a trashcan directory and are purged only after the configured `fs.trash.interval` (the trashcan is disabled by default in Apache Hadoop, value `0`; many distributions enable it and set an interval). Use `DROP TABLE ... PURGE` to skip the trashcan and delete immediately.

For external tables the safe cleanup is: drop the table, then remove the HDFS path explicitly once you are certain nothing else references it.

**Mental model.** `DROP TABLE` deletes data files only for managed (internal) tables. External tables own only the metadata, so their files stay where they are. `DESCRIBE FORMATTED` shows `MANAGED_TABLE` or `EXTERNAL_TABLE` in the `Table Type` row. Even for managed tables, files usually go to the HDFS trash and are purged after `fs.trash.interval`.

**Worked example and common mistake.**

```sql
-- Check before dropping, then decide how to clean up
describe formatted analytics.fact_orders;

-- Managed table: skip the trashcan and reclaim space immediately
drop table if exists analytics.fact_orders purge;

-- External table: drop metadata, then remove the path yourself
drop table if exists analytics.orders_ext;
-- hdfs dfs -rm -r -skipTrash /warehouse/analytics/orders_ext
```

Common mistake: dropping the external table and assuming the space is gone. Verify the location with `DESCRIBE FORMATTED`, then remove the directory explicitly once nothing else uses it.

## How do I load a big Csv file into a partitioned table?

To load a data file into a partitioned table, when the data file includes fields like year, month, and so on that correspond to the partition key columns, use a two-stage process. First, use the `LOAD DATA` or `CREATE EXTERNAL TABLE` statement to bring the data into an unpartitioned text table. Then use an `INSERT ... SELECT` statement to copy the data from the unpartitioned table to a partitioned one. Include a `PARTITION` clause in the INSERT statement to specify the partition key columns.

```sql
-- Stage 1: land the raw CSV as an external text table
create external table staging.orders_csv (
  order_id string,
  customer_id string,
  amount string,
  order_date string
)
row format delimited fields terminated by ','
location '/data/incoming/orders/2026-10-04/';

-- Stage 2: typed, partitioned, columnar target
create table analytics.fact_orders (
  order_id bigint,
  customer_id bigint,
  amount decimal(12,2)
)
partitioned by (order_date date)
stored as parquet;

-- Dynamic partition insert: one pass, one directory per date
set num_dynamic_partitions = 366;
insert into analytics.fact_orders partition (order_date)
select cast(order_id as bigint), cast(customer_id as bigint),
       cast(amount as decimal(12,2)), cast(order_date as date)
from staging.orders_csv;

compute stats analytics.fact_orders;
```

For very large files, load in date ranges to keep the number of open files and the memory for dynamic partitions under control.

**Mental model.** Text CSV is a landing format, not a query format. Stage it as an external text table over the incoming directory, then `INSERT ... SELECT` into a typed partitioned Parquet table. The `PARTITION` clause maps the last projected columns to partition keys. Dynamic partitioning creates one directory per distinct value in one pass.

**Gotcha.** CSV files often carry a header row and untyped nulls, and a single `INSERT` over many partition values can exhaust memory. Handle the header with a table property, cast explicitly, and load in bounded date ranges.

```sql
create external table staging.orders_csv (
  order_id string, customer_id string, amount string, order_date string
)
row format delimited fields terminated by ','
tblproperties ('skip.header.line.count' = '1')
location '/data/incoming/orders/';

set num_dynamic_partitions = 400;
insert into analytics.fact_orders partition (order_date)
select cast(order_id as bigint), cast(customer_id as bigint),
       cast(amount as decimal(12,2)), cast(order_date as date)
from staging.orders_csv
where order_date between '2026-01-01' and '2026-01-31';

compute stats analytics.fact_orders;
```

Common mistake: loading everything in one statement, then discovering that the header row became a row with null keys and that the insert exceeded memory.

## Can I Do Insert Select Into a partitioned table?

When you use the `INSERT ... SELECT *` syntax to copy data into a partitioned table, the columns corresponding to the partition key columns must appear last in the columns returned by the `SELECT *`. You can create the table with the partition key columns defined last. Or, you can use the CREATE VIEW statement to create a view that reorders the columns: put the partition key columns last, then do the `INSERT ... SELECT *` from the view.

Explicit column lists are clearer and avoid the ordering trap entirely:

```sql
insert into analytics.fact_orders (order_id, customer_id, amount, order_date)
select order_id, customer_id, amount, order_date
from staging.orders_clean;
```

**Mental model.** In a positional `INSERT ... SELECT`, the projected columns must line up with the target table's columns, and the partition key columns come last because they are not stored inside the data files. An explicit column list in the `INSERT` removes the ordering requirement entirely.

**Worked example and common mistake.**

```sql
-- Explicit and order-independent
insert into analytics.fact_orders (order_id, customer_id, amount, order_date)
select order_id, customer_id, amount, order_date
from staging.orders_clean;
```

Common mistake: `INSERT ... SELECT *` where the source puts the partition key in the middle. The insert either fails with a column-count mismatch or writes the wrong value into the wrong column. Use the explicit list or a view that reorders columns.

## How do you optimize Impala queries?

Use Parquet, collect table/column statistics, filter partition columns, avoid tiny files, and check the query profile for scan and join bottlenecks. Join order and broadcast vs partitioned joins can matter for large tables.

Example:

```sql
compute stats fact_orders;
```

Statistics help the optimizer estimate cardinality and choose better join plans.

A practical checklist:

+ Project only needed columns; Impala is columnar, so `select *` on a wide table is expensive.
+ Filter on partition columns so partitions are pruned.
+ Run `COMPUTE STATS` (or `COMPUTE INCREMENTAL STATS`) after loads.
+ Watch for `BROADCAST` joins on large tables and for exchange row counts in `explain`.
+ Compact small files and keep files in the 128-512 MB range.
+ Use `COMPUTE STATS` on both sides of a join, not just the large table.

**Mental model.** Optimize in this order: format and layout, then statistics, then the plan, then memory. Format and layout decide how much data is read. Statistics decide whether the plan is good. The plan decides how much data moves. Memory and admission control decide whether the query runs at all under concurrency.

**Worked example.** A dashboard query was scanning three years of orders because the filter used `cast(order_date as string) = '2026-10-04'` instead of a date predicate. Rewriting it as `order_date = '2026-10-04'` restored partition pruning, and `COMPUTE STATS` let the optimizer pick a broadcast join for the small dimension.

```sql
compute stats analytics.fact_orders;
compute incremental stats analytics.fact_orders partition (order_date = '2026-10-04');

explain
select c.region, count(*) as orders, sum(o.amount) as revenue
from analytics.fact_orders o
join analytics.dim_customer c on o.customer_id = c.customer_id
where o.order_date = '2026-10-04'
group by c.region;
```

Common mistake: tuning the query before fixing the layout. No hint makes a million tiny files fast, and no plan helps when the filter prevents partition pruning.

## How do you create Impala tables with DDL and choose a file format?

Impala DDL is Hive-compatible; the key choices are internal vs external, file format, and partition layout.

```sql
create table if not exists analytics.fact_orders (
  order_id bigint, customer_id bigint, status string,
  amount decimal(12,2), order_ts timestamp
)
partitioned by (order_date date)
stored as parquet
tblproperties ('parquet.compression' = 'snappy');

create external table if not exists analytics.orders_ext (
  order_id bigint, customer_id bigint, amount decimal(12,2)
)
partitioned by (order_date date)
stored as parquet
location '/warehouse/analytics/orders_ext';

-- Change compression for future writes
alter table analytics.fact_orders set tblproperties ('parquet.compression' = 'zstd');
```

Format guidance: Parquet for analytical scans, ORC where the cluster is Hive-centric, Avro/Text for landing only, and Kudu for mutable key-addressable tables.

**Mental model.** File format determines scan cost and capabilities. Parquet is columnar with per-row-group min/max statistics and dictionary encoding, which enables projection and predicate pushdown; ORC is similar and common in Hive-centric clusters. Avro and text are row-oriented landing formats. Kudu is a mutable, primary-key table format rather than a file format. Iceberg adds table-level snapshots on top of Parquet or ORC files.

**How pushdown works.** Impala reads only the referenced columns, then skips row groups whose statistics prove that no row can match the predicate. A function wrapped around a column hides the comparison, so `cast(date_col as string) = '2026-10-04'` defeats pushdown where `date_col = '2026-10-04'` uses it.

```sql
create table analytics.fact_orders (
  order_id bigint, customer_id bigint, status string,
  amount decimal(12,2), order_ts timestamp
)
partitioned by (order_date date)
stored as parquet
tblproperties ('parquet.compression' = 'zstd');

-- Future writes use the new codec; existing files keep their own
alter table analytics.fact_orders set tblproperties ('parquet.compression' = 'zstd');

-- Check what a table actually is
describe formatted analytics.fact_orders;
```

Common mistake: choosing a format for storage reasons only. Text compresses poorly and cannot be pushed down, so it should never be the format analysts query repeatedly.

## How do you create partitioned tables and manage partitions in Impala?

Partitioning is the highest-leverage decision in Impala because it enables partition pruning, so a date-filtered query reads only matching directories.

```sql
create table analytics.fact_events (
  event_id bigint, user_id bigint, event_ts timestamp, payload string
)
partitioned by (event_month string, event_day string)
stored as parquet;

alter table analytics.fact_events
  add partition (event_month = '2026-10', event_day = '2026-10-04');

set num_dynamic_partitions = 400;
insert into analytics.fact_events partition (event_month, event_day)
select event_id, user_id, event_ts, payload,
       date_format(event_ts, 'yyyy-MM'), date_format(event_ts, 'yyyy-MM-dd')
from staging.events_raw;

alter table analytics.fact_events drop partition (event_month = '2025-01');
```

Keep partition count in the thousands, not millions, and never partition by a high-cardinality key such as `user_id` or a raw timestamp.

**Mental model.** A partitioned table stores one directory per partition value, and the partition columns are encoded in the directory path rather than in the files. The planner prunes directories using the predicates in the query, so a filter on a partition column removes entire directories from the scan before any file is opened.

**How partitions get created.** Static partitions are named explicitly with `ALTER TABLE ... ADD PARTITION` or a literal `PARTITION` clause in an insert. Dynamic partitions come from the last projected columns of an insert and are created on the fly; `num_dynamic_partitions` caps how many one statement may create. Use dynamic for bulk loads and static for controlled backfills.

```sql
-- Add one partition explicitly
alter table analytics.fact_events
  add partition (event_month = '2026-10', event_day = '2026-10-04');

-- Dynamic insert: one directory per month/day in the input
set num_dynamic_partitions = 400;
insert into analytics.fact_events partition (event_month, event_day)
select event_id, user_id, event_ts, payload,
       date_format(event_ts, 'yyyy-MM'), date_format(event_ts, 'yyyy-MM-dd')
from staging.events_raw;

-- Dropping a partition removes its directory for managed tables
alter table analytics.fact_events drop partition (event_month = '2025-01');
```

Common mistake: partitioning by a high-cardinality column such as `user_id` or a raw timestamp. Thousands of partitions are manageable; millions produce metadata bloat, slow planning, and thousands of tiny files.

## How do COMPUTE STATS and INVALIDATE METADATA drive Impala query plans?

Impala's optimizer is cost-based, so it needs statistics to choose join order and strategy. Without them it guesses, which is the root cause of most "Impala picked a terrible plan" incidents.

```sql
compute stats analytics.fact_orders;
compute incremental stats analytics.fact_orders partition (order_date = '2026-10-04');
show table stats analytics.fact_orders;
show column stats analytics.fact_orders;
refresh analytics.fact_orders;
invalidate metadata analytics.fact_orders;
```

`REFRESH` re-reads data/file metadata; `INVALIDATE METADATA` re-reads catalog structure. Scope `INVALIDATE METADATA` to one table and run it off-peak, because it is expensive at the NameNode.

**Mental model.** The optimizer is cost-based and needs cardinality estimates. `COMPUTE STATS` scans the table once and writes row counts, file counts, sizes, and per-column statistics (number of distinct values, nulls, average and max size) into the metastore. The planner then estimates filter selectivity and join sizes from those numbers.

**Full versus incremental.** `COMPUTE STATS` rescans the whole table. `COMPUTE INCREMENTAL STATS` maintains per-partition statistics and only rescanning changed partitions, so on a large partitioned table it turns a multi-hour job into a per-partition one. Incremental stats must be enabled by running the full command once; after that, `DROP STATS` removes them. If a partition has no incremental stat, the planner falls back to the table-level estimate.

```sql
-- One-time full scan, then per-partition maintenance
compute stats analytics.fact_orders;
compute incremental stats analytics.fact_orders partition (order_date = '2026-10-04');

show table stats analytics.fact_orders;
show column stats analytics.fact_orders;

-- Remove stale stats after a schema change
drop stats analytics.fact_orders;
```

Common mistake: collecting stats only on the fact table. The dimension side also feeds the estimate, so a missing `COMPUTE STATS` on the small table can leave the planner guessing and broadcasting the wrong side.

## How do you read an Impala EXPLAIN plan and tune joins and hints?

`EXPLAIN` shows the physical plan with estimates; `profile` shows what actually happened. Read them together.

```sql
explain
select c.region, count(*) as orders, sum(o.amount) as revenue
from analytics.fact_orders o
join analytics.dim_customer c on o.customer_id = c.customer_id
where o.order_date >= '2026-10-01'
group by c.region;
```

What to look for: `PARTITION RANGE` proves pruning is working; `HASH JOIN [BROADCAST]` on a large right side is the classic memory/network mistake; `EXCHANGE` rows quantify network cost; estimate-vs-actual gaps mean stale statistics.

```sql
select /* +BROADCAST */ ... from small_dim d join big_fact f on d.id = f.dim_id;
select /* +SHUFFLE */    ... from big_fact a join big_fact_b b on a.k = b.k;
select /* +STRAIGHT_JOIN */ ... from dim d join fact f on d.id = f.dim_id;
set runtime_filter_mode = global;   -- global | per_node | off
```

**Mental model.** `EXPLAIN` prints the physical plan tree with estimated cardinality and, for joins, the chosen distribution. `PROFILE` (or the query profile UI) prints what actually happened, operator by operator, including rows produced, peak memory, and spill. Read the estimate first, then the reality, and compare.

**How to read it.** Nodes are numbered and indented under `PLAN-ROOT SINK`. `SCAN HDFS` lines show `partitions=`, `files=`, and `size=`. `HASH JOIN` lines show `[BROADCAST]` or `[PARTITIONED]` and the join predicates. `EXCHANGE` shows how rows move between fragments. In the profile, compare estimated versus actual rows per operator, then look at `TotalTime`, `PeakMemoryUsage`, `RowsSpilled`, and `BytesSpilled`.

```sql
set explain_level = 3;   -- more detail on expressions and predicates

explain
select c.region, count(*) as orders
from analytics.fact_orders o
join analytics.dim_customer c on o.customer_id = c.customer_id
where o.order_date = '2026-10-04'
group by c.region;
```

Common mistake: reading `EXPLAIN` top-down and stopping at the first join. Start at the scan lines to confirm pruning, then follow the rows upward to find the operator whose output is far smaller than its input, which is where the real work happens.

## How do you use Parquet and Kudu tables in Impala?

Parquet is the immutable, cheap-to-scan columnar format for facts; Kudu is the mutable, primary-key store for dimensions and operational lookups.

```sql
create table analytics.customer_kudu (
  customer_id bigint, name string, tier string, updated_at timestamp,
  primary key (customer_id)
)
partition by hash (customer_id) partitions 16
stored as kudu;

upsert into analytics.customer_kudu (customer_id, name, tier, updated_at)
values (1001, 'Ada Lovelace', 'gold', now());

update analytics.customer_kudu set tier = 'platinum' where customer_id = 1001;
delete from analytics.customer_kudu where customer_id = 1002;
```

Choose Kudu for upserts and millisecond key lookups, and Parquet when you need the cheapest large-scale scans.

**Mental model.** Parquet tables are directories of immutable files optimized for large scans. Kudu tables are live distributed tables split into tablets by a partition schema, replicated for durability, and addressable by primary key. Kudu keeps rows in a columnar on-disk format with an in-memory delta store, so both key lookups and range scans are fast.

**How the partition schema works.** Kudu distributes rows by hash partitions on key columns to spread writes, and by range partitions on a key column to bound scans and allow cheap range drops. A well-designed schema hashes on the first key column and ranges on a time column. Kudu rows are updated in place, so `UPDATE`, `DELETE`, and `UPSERT` are cheap when the primary key is fully specified.

```sql
create table analytics.customer_kudu (
  customer_id bigint, name string, tier string, updated_at timestamp,
  primary key (customer_id)
)
partition by hash (customer_id) partitions 16
stored as kudu;

upsert into analytics.customer_kudu (customer_id, name, tier, updated_at)
values (1001, 'Ada Lovelace', 'gold', now());

update analytics.customer_kudu set tier = 'platinum' where customer_id = 1001;
```

Common mistake: putting multi-billion-row fact data in Kudu because upserts are convenient. Kudu capacity is bounded by local disks on its tablet servers and it is not the cheapest place for large analytical scans; keep facts in Parquet and dimensions in Kudu.

## How do Impala resource pools and admission control work?

Admission control decides whether a query starts immediately, waits in a queue, or is rejected. It is configured per resource pool and enforced by `impalad`.

```text
<queue name="root">
  <queue name="bi">
    <maxResources>40000 mb</maxResources>
    <maxRunningApps>20</maxRunningApps>
  </queue>
  <queue name="etl">
    <maxResources>80000 mb</maxResources>
    <maxRunningApps>5</maxRunningApps>
  </queue>
</queue>
```

```sql
set request_pool = 'bi';
set mem_limit = 16gb;
-- Check retries/attempts in the runtime profile (impala-shell `PROFILE`, or the query profile UI)
```

Give interactive BI a shallow, small-memory pool so a huge ETL query cannot block dashboards, and use `mem_limit` as a second per-query cap.

**Mental model.** Admission control is a gate in front of query execution. The planner estimates the memory a query needs, then the coordinator checks the target pool's limits against what is already running. The query starts immediately, waits in the pool queue, or is rejected. Pools are defined in the fair-scheduler XML and referenced per session or per query with `REQUEST_POOL`.

**How it is enforced.** Each pool has `maxResources` (memory and CPU) and `maxRunningApps` (concurrent queries). A query also has its own `MEM_LIMIT`. Waiting is bounded by `RESERVATION_REQUEST_TIMEOUT`; if no slot frees up in time, the query fails rather than waiting forever. `impalad` re-estimates at runtime and can cancel a query that exceeds its limit.

```text
<queue name="root">
  <queue name="bi">
    <maxResources>40000 mb</maxResources>
    <maxRunningApps>20</maxRunningApps>
  </queue>
  <queue name="etl">
    <maxResources>80000 mb</maxResources>
    <maxRunningApps>5</maxRunningApps>
  </queue>
</queue>
```

```sql
set request_pool = 'bi';
set mem_limit = 16gb;
set reservation_request_timeout = 30000;   -- milliseconds to wait for a slot
```

Common mistake: one shared pool for dashboards and ETL. A single heavy ETL query then fills the pool and queues every interactive user. Separate pools with independent caps keep the SLAs apart.

## How do you handle small files and table maintenance in Impala?

Small files are the most common Impala performance problem. Each file adds planning and open cost, and thousands of tiny files per partition slow the NameNode and every scan.

```sql
set PARQUET_FILE_SIZE = 512m;
insert overwrite analytics.fact_orders partition (order_date = '2026-10-04')
select order_id, customer_id, status, amount, order_ts
from analytics.fact_orders where order_date = '2026-10-04';

refresh analytics.fact_orders partition (order_date = '2026-10-04');
compute stats analytics.fact_orders;
```

Prevention beats cleanup: land 128-512 MB files from Spark/Hive and prefer partition overwrite over many tiny appends.

**Mental model.** Every file adds planning work, a NameNode lookup, and an open/seek/read cost. A partition with ten 200 MB files is efficient; a partition with ten thousand 200 KB files spends more time in metadata than in scanning. Small files usually come from streaming micro-batches or many concurrent inserts into dynamic partitions.

**How to fix and prevent it.** Rewrite the partition through `INSERT OVERWRITE` with a larger `PARQUET_FILE_SIZE`, which compacts in one pass, then `REFRESH` and collect stats. Prevent recurrence by writing 128-512 MB files from the upstream job, preferring partition overwrite to appends, and running periodic compaction on the busiest partitions.

```sql
set parquet_file_size = 512m;

insert overwrite analytics.fact_orders partition (order_date = '2026-10-04')
select order_id, customer_id, status, amount, order_ts
from analytics.fact_orders where order_date = '2026-10-04';

refresh analytics.fact_orders partition (order_date = '2026-10-04');
compute incremental stats analytics.fact_orders partition (order_date = '2026-10-04');
```

Common mistake: compacting without updating statistics. The file count changes, and the planner keeps using the old numbers until stats are recomputed.

## What are Impala's trade-offs and when should you not use it?

+ **Performance:** extremely fast for MPP scans and joins over Parquet/Kudu, but not a transactional warehouse; no multi-statement transactions, and heavy ETL belongs on Spark.
+ **Resource contention:** memory-hungry. Broadcast joins and high-cardinality aggregations can exhaust a node, so admission control and `mem_limit` are mandatory.
+ **Exactly-once / mutability:** plain Parquet cannot do single-row updates, so use `INSERT OVERWRITE` of a partition as the idempotent pattern; Kudu adds upserts but also an operational component.
+ **Concurrency:** dozens to low hundreds of concurrent queries are fine, thousands are not.

When NOT to use Impala: OLTP writes (use PostgreSQL/Kudu-backed services), long multi-stage shuffles (Spark/Hive/Tez), sub-second stateful streaming (Flink), and text search (a search engine).

**Mental model.** Impala trades flexibility for scan speed. It has no multi-statement transactions on HDFS tables, no secondary indexes, a small set of writable formats, and memory-hungry operators. In exchange it gives predictable interactive latency over columnar data with a mature planner and admission control.

**Gotcha.** The trade-off that surprises teams most is concurrency versus query size. Impala is happy with many small queries or a few large ones, but mixing them without pools makes both slow. Decide the split before launch; retrofitting pools after an incident is harder.

## Tuning 1: statistics and metadata for good plans

**Mental model.** The Impala planner is cost-based, so every plan decision rests on cardinality estimates. Those estimates come from table and column statistics stored in the metastore. With no statistics, the planner falls back to guesses about row count and column distinct values, and guesses are exactly what produce broadcast joins on large tables, wrong join order, and bad exchange sizing.

**How the statistics work.** `COMPUTE STATS` scans the table once and records the row count, file count, total size, and per-column number of distinct values, null count, average size, and maximum size. The planner uses those to estimate filter selectivity and the size of each join input. `COMPUTE INCREMENTAL STATS` stores the same numbers per partition and only rescans partitions whose statistics are missing, so after the first full run you maintain a huge partitioned table one partition at a time. `SHOW TABLE STATS` and `SHOW COLUMN STATS` reveal what the planner actually knows.

**Common mistake.** Loading a new partition and skipping statistics. The planner keeps using the old row count, decides the table is smaller than it is, and broadcasts it. A tell-tale sign is a `BROADCAST` join on a table that has grown for months while `SHOW TABLE STATS` still reports the row count from the last full collection. Refresh the metadata after external writes, then recompute the affected statistics.

```sql
-- Full collection once, then maintain per partition
compute stats analytics.fact_orders;
compute incremental stats analytics.fact_orders partition (order_date = '2026-10-04');

-- Inspect what the planner believes
show table stats analytics.fact_orders;
show column stats analytics.fact_orders;
```

In [ ]:
# Estimate the cost of a full versus an incremental statistics collection.
rows = 4_000_000_000
avg_row_bytes = 120
row_groups_per_second = 4_000_000      # cluster scan rate for COMPUTE STATS
partitions = 3 * 365
table_bytes = rows * avg_row_bytes
table_gb = table_bytes / 1024 ** 3

full_seconds = rows / row_groups_per_second
partition_seconds = full_seconds / partitions

print(f"table size      = {table_gb:,.0f} GB")
print(f"full stats      = {full_seconds / 60:,.1f} minutes")
print(f"one partition   = {partition_seconds:.2f} seconds")

assert full_seconds / 60 > 10
assert partition_seconds < 1
print("verdict: collect full stats once, then incremental per load")


## Tuning 2: reading EXPLAIN and PROFILE to find the slow operator

**Mental model.** `EXPLAIN` shows the physical plan with estimates. The profile shows actual execution counters per operator. The gap between the two is the story: an estimated 50 thousand rows that turned out to be 2.4 million is a cardinality misestimate, and it usually explains a queue of other symptoms such as spill and a slow exchange.

**How to read the plan.** Start at the `SCAN HDFS` lines and confirm pruning with `partitions=` and `files=`. Move upward and check each `HASH JOIN` for `[BROADCAST]` versus `[PARTITIONED]`. `EXCHANGE` nodes are where data moves between fragments, so their row counts are network cost. In the profile, compare estimated with actual rows per operator and look at `TotalTime`, `PeakMemoryUsage`, `RowsSpilled`, and `BytesSpilled`.

```text
# Abbreviated shape of a partitioned join plan
03:EXCHANGE [UNPARTITIONED]
02:HASH JOIN [INNER JOIN, PARTITIONED]
|  hash predicates: o.customer_id = c.customer_id
|  runtime filters: RF000 <- c.customer_id
|--01:SCAN HDFS [analytics.dim_customer c]  partitions=1/1 files=4 size=12MB
00:SCAN HDFS [analytics.fact_orders o]      partitions=1/365 files=96 size=1.9GB
    predicates: o.order_date = '2026-10-04'
```

In [ ]:
# Rank operators by time, then flag cardinality misestimates and spill.
operators = [
    {"name": "SCAN HDFS fact_orders", "estimated_rows": 12_000_000, "actual_rows": 12_400_000,
     "total_time_s": 4.2, "rows_spilled": 0},
    {"name": "HASH JOIN dim_customer", "estimated_rows": 50_000, "actual_rows": 2_400_000,
     "total_time_s": 31.5, "rows_spilled": 1_800_000},
    {"name": "AGGREGATE finalize", "estimated_rows": 8, "actual_rows": 8,
     "total_time_s": 0.4, "rows_spilled": 0},
]

worst = max(operators, key=lambda op: op["total_time_s"])
print("slowest operator:", worst["name"], f'{worst["total_time_s"]}s')

flagged = []
for op in operators:
    ratio = op["actual_rows"] / max(op["estimated_rows"], 1)
    if ratio >= 10:
        flagged.append(op["name"])
        print(f'misestimate x{ratio:,.0f} on {op["name"]}')

assert worst["name"] == "HASH JOIN dim_customer"
assert flagged == ["HASH JOIN dim_customer"]
assert worst["rows_spilled"] > 0


**Common mistake.** Blaming the scan because it is the largest node in the tree. Scans usually stream steadily and cheaply; the expensive operator is the one where output cardinality collapses or explodes. Find the operator whose `actual_rows` diverges most from its estimate, then fix the statistics, the predicate, or the join strategy behind it.

## Tuning 3: join and file layout tuning

**Mental model.** Join strategy decides where the data moves. A broadcast join copies the build side to every worker, which is cheap for a small dimension and fatal for a large table. A partitioned join shuffles both sides on the join key, which scales but pays network and disk. The planner chooses from statistics, and hints exist only to override a known bad choice.

**Runtime filters.** When Impala builds the hash table for a join, it also builds a filter on the join key (a bloom filter plus min/max bounds). In `global` mode that filter is routed to every probe-side scan fragment, so the probe side drops non-matching rows before they reach the join and before they cross an exchange. `SET RUNTIME_FILTER_MODE=global` is the default in modern Impala; `per_node` limits routing to nodes running both fragments, and `off` disables it.

**File layout.** Partition pruning removes directories; file sizing removes planning and open overhead. Aim for 128-512 MB Parquet files per partition, keep the partition count in the thousands, and compact partitions that have drifted into small files. The three levers work together: a partitioned table with right-sized files and current statistics lets the plan run close to disk speed.

```sql
set runtime_filter_mode = global;
set runtime_filter_wait_time_ms = 1000;
set parquet_file_size = 256m;
```

In [ ]:
# Decide the join strategy from build-side size, then size the files.
BUILD_MEMORY_BUDGET_MB = 512
TARGET_FILE_MB = 256

small_rows, small_bytes_per_row = 2_000_000, 80
fact_rows = 4_000_000_000
build_mb = small_rows * small_bytes_per_row / 1024 ** 2

strategy = "broadcast" if build_mb <= BUILD_MEMORY_BUDGET_MB else "partitioned"
print(f"build side = {build_mb:,.0f} MB -> {strategy}")

partition_gb = 190
files = -(-partition_gb * 1024 // TARGET_FILE_MB)   # ceiling division
avg_file_mb = partition_gb * 1024 / files
print(f"partition {partition_gb} GB -> {files} files, {avg_file_mb:.0f} MB each")

assert build_mb == 152.587890625
assert strategy == "broadcast"
assert files == 760
assert 128 <= avg_file_mb <= 512


**Common mistake.** Forcing a broadcast hint to avoid a shuffle. The hint fixes one query and creates a memory incident the next time the dimension grows. Fix the statistics and the file layout first; use `/* +BROADCAST */` or `/* +SHUFFLE */` as a temporary override with a note in the query.

## Tuning 4: memory, spill and admission control

**Mental model.** Memory is bounded twice: per query by `MEM_LIMIT`, and per pool by admission control. When an operator exceeds its usable memory it spills to local scratch disk, if the operator supports spilling. `RowsSpilled` and `BytesSpilled` in the profile are the honest signal that a query is memory-starved even when it finishes.

**Admission control and codegen.** Pools cap concurrency and total memory so one heavy query cannot starve dashboards. `REQUEST_POOL` picks the pool, `RESERVATION_REQUEST_TIMEOUT` bounds how long a query waits for a slot, and `MEM_LIMIT` is the per-query ceiling. `SET DISABLE_CODEGEN=true` is a diagnostic for CPU-specific failures, not a tuning knob. On Kudu tables, remember that scans compete with the tablet server's own memory and that `UPDATE`/`DELETE` need full primary-key equality predicates.

In [ ]:
# Decide whether an operator will spill and how much scratch it needs.
MEM_LIMIT_GB = 32
SCRATCH_LIMIT_GB = 100
operators = {"hash join build": 40, "aggregate": 12, "sort": 18}

needs_spill = {name: mem for name, mem in operators.items() if mem > MEM_LIMIT_GB}
scratch_gb = sum(mem - MEM_LIMIT_GB for mem in needs_spill.values())
print("must spill:", sorted(needs_spill))
print(f"estimated scratch = {scratch_gb} GB")

assert list(needs_spill) == ["hash join build"]
assert scratch_gb == 8
assert scratch_gb < SCRATCH_LIMIT_GB
print("verdict: raise MEM_LIMIT or switch the build side to a shuffle join")


**Common mistake.** Raising `MEM_LIMIT` for every failing query until the node is oversubscribed. Spill is often a plan problem: a broadcast build side that should be shuffled, an aggregation key that is too granular, or a dynamic partition insert writing too many partitions at once. Fix the shape before buying memory.

## Things to remember: key numbers and limits


| Topic | Fact | Why it matters |
|---|---|---|
| Architecture | every query runs as an MPP fragment graph across impalad daemons; no MapReduce | low-latency interactive SQL comes from skipping the MR batch machinery — and losing its fault tolerance |
| Catalog daemons | statestored pushes catalog/metadata deltas; catalogd owns the metadata | `INVALIDATE METADATA` (catalog changed) vs `REFRESH` (data files changed) is the daily operator reflex |
| Statistics | the CBO plans from `COMPUTE STATS` output | a table loaded without stats gets the wrong join strategy silently |
| Joins | broadcast (small build side replicated) vs partitioned (both sides shuffled) | the misestimate that matters most is the build side exceeding broadcast range |
| Memory | aggregation/join spilling; admission control queues cap concurrent memory | query failure under load is usually admission control, not bad SQL |
| File layout | Parquet is the default; many small files hurt planning and scan | compact on ingest, partition for pruning |
| Mutability | HDFS/Parquet tables are bulk-optimized (no per-row UPDATE); Kudu is the mutable counterpart | point updates mean you chose Kudu, or you rewrite partitions |


## Things to remember: semantics to memorize


* **Interactive, not batch.** Impala answers in seconds by holding a query's fragments in memory across daemons — a mid-query node failure kills the query, which is why long ETL stayed on Hive.
* **Metadata is cached aggressively.** Writers creating files outside Impala must be followed by REFRESH/INVALIDATE METADATA, or Impala plans against a stale catalog — the number-one 'my new data doesn't appear' cause.
* **Partition pruning happens against the directory tree.** A query without a predicate on the partition column scans every file.
* **Stats are per-column and per-partition.** Incremental stats keep ANALYZE cheap on growing tables; stale NDV counts flip join order.
* **Spill-to-disk saves big joins from failing — at a price.** Scratch usage in PROFILE is the tell that a query is one size up from its memory envelope.


## Things to remember: mistakes that keep coming back


* Forgetting `REFRESH` after an external load — the table looks empty or stale in Impala while Hive sees the data fine.
* Skipping `COMPUTE STATS` after bulk loads — broadcast vs partitioned decisions go wrong and giant tables get broadcast.
* Broadcasting an over-large build side — switch strategy or stats first, not more memory.
* Thousands of small files from micro-batch writes — compaction jobs and `INSERT INTO` rewrites are the fix, not more daemons.
* Assuming UPDATE works on a Parquet table — it does not; use Kudu or rewrite the partition.
* Letting one huge query eat the cluster — admission control pools exist to keep interactive SLAs alive under batch load.


## Real world use cases for Apache Impala

### 1. Interactive BI over a partitioned Parquet lake

A retailer's analysts query three years of orders because Hive reports take minutes. ~2 PB in HDFS, ~4 billion rows, daily partitions. Spark writes Parquet; Impala serves Hue/Tableau; `catalogd` broadcasts metadata after ETL.

```sql
compute stats analytics.fact_orders;

select order_date, sum(amount) as revenue
from analytics.fact_orders
where order_date between '2026-09-01' and '2026-09-30'
group by order_date order by order_date;
```

Why: pruning keeps a month-scoped query off 97% of the data, and statistics let the optimizer choose a sane plan without hints.

### 2. Operational dimension store on Kudu

A subscription product needs a customer table updated continuously by a service while analysts join it from SQL. ~50 million customers, ~20k upserts/minute. The service writes Kudu; Parquet facts join the Kudu dimension.

```sql
upsert into analytics.dim_customer (customer_id, email, plan, updated_at)
values (1001, 'ada@example.com', 'pro', now());

select customer_id, plan from analytics.dim_customer where customer_id = 1001;
```

Why: primary-key upserts and millisecond lookups, plus SQL access to the same mutable table without a second copy.

### 3. Ad-hoc exploration for a data science team

Data scientists slice clickstream data without waiting for Spark scheduling. ~800 TB, ~120 billion events, hourly partitions on the busiest flows, a dedicated pool, nightly compaction.

```sql
set request_pool = 'ds_ad_hoc';
set mem_limit = 32gb;

select device_type, country, count(*) as events
from analytics.fact_events
where event_date = '2026-10-04' and event_hour between '09' and '17'
group by device_type, country order by events desc limit 50;
```

Why: an isolated pool stops one exploratory query blocking BI, and hour-level pruning makes a huge table interactive.

### 4. Log analytics with HBase for sparse lookups

A support tool fetches all attributes for one user from a wide, sparse profile table. ~300 million profiles with hundreds of optional columns. HBase holds the profile; Impala serves key lookups and Parquet serves dashboards.

```sql
select customer_id, registered_at, last_login, loyalty_tier
from hbase.customer_profile
where customer_id = 'C-884422';
```

Why: HBase suits a few rows by key with a sparse schema, while full-scan analytics stays on columnar Parquet.

### 5. Migrating Hive batch reports to interactive SQL

A telco wants on-demand versions of years of Hive reports. ~500 TB of ORC/Parquet. Hive/Spark stays the ETL writer, Impala coordinators share its metastore, and metadata is refreshed after each ETL run.

```sql
refresh analytics.monthly_revenue;
compute stats analytics.monthly_revenue;
select * from analytics.monthly_revenue where month = '2026-09' order by revenue desc limit 20;
```

Why: reusing the metastore avoids duplicating data, and `REFRESH` plus `COMPUTE STATS` keeps Impala correct without a migration project.

**How to answer these in an interview.** For each scenario, name the storage format, the partition key, the join strategy, and the guardrail. A strong answer sounds like: Parquet facts partitioned by day, ZSTD compression, 256 MB files, incremental stats after each load, Kudu for the mutable dimension, broadcast joins for the small dimension, and a dedicated BI pool with `MEM_LIMIT` caps.

**Common mistake across all five cases.** Describing only the SQL and skipping the operational envelope. Impala designs fail on statistics, file sizing, and admission control far more often than on query syntax, so state those explicitly alongside the query.

## Impala exercises

### Try it 1. Broadcast or partitioned join?

**Task.** A 2-million-row dimension is 80 bytes per row. The fact table has 4 billion rows. Broadcast is accepted when the estimated build side is at most 512 MB; otherwise use a partitioned join. Compute the build-side size and choose the strategy.

**Hint.** `build_mb = rows * bytes_per_row / 1024 ** 2`, then compare with the budget.

**Expected result.** Build side `152.6 MB`, so the verdict is `broadcast`.

In [ ]:
BUILD_BUDGET_MB = 512
dim_rows, dim_bytes = 2_000_000, 80
fact_rows = 4_000_000_000

build_mb = dim_rows * dim_bytes / 1024 ** 2
strategy = "broadcast" if build_mb <= BUILD_BUDGET_MB else "partitioned"

print(f"dimension build side = {build_mb:,.1f} MB")
print(f"fact rows = {fact_rows:,} -> strategy: {strategy}")

assert round(build_mb, 1) == 152.6
assert strategy == "broadcast"


### Try it 2. Detect a cardinality misestimate

**Task.** Given estimated and actual row counts for three operators, compute the actual/estimated ratio and flag every operator whose ratio is at least 10. Report the flagged operator and its ratio.

**Hint.** Use `actual / max(estimated, 1)` so a zero estimate does not divide by zero.

**Expected result.** One flag: `HASH JOIN dim_customer` with a ratio of `48.0`.

In [ ]:
operators = [
    ("SCAN HDFS fact_orders", 12_000_000, 12_400_000),
    ("HASH JOIN dim_customer", 50_000, 2_400_000),
    ("AGGREGATE finalize", 8, 8),
]

flagged = []
for name, estimated, actual in operators:
    ratio = actual / max(estimated, 1)
    print(f"{name:<24} estimated={estimated:>10,} actual={actual:>10,} ratio={ratio:6.1f}")
    if ratio >= 10:
        flagged.append((name, round(ratio, 1)))

assert flagged == [("HASH JOIN dim_customer", 48.0)]
print("misestimated operators:", flagged)


### Try it 3. File count for a target size

**Task.** A partition holds 190 GB. Compute how many files it should contain if the target Parquet file size is 256 MB, then compute the average file size. Flag the layout as small-file-bound when the average is below 128 MB.

**Hint.** Use ceiling division: `-(-total_mb // target_mb)`.

**Expected result.** `760` files at `256.0 MB` each, so the layout is healthy.

In [ ]:
partition_gb = 190
TARGET_FILE_MB = 256
SMALL_FILE_MB = 128

total_mb = partition_gb * 1024
files = -(-total_mb // TARGET_FILE_MB)
avg_file_mb = total_mb / files
small_file_bound = avg_file_mb < SMALL_FILE_MB

print(f"{partition_gb} GB / {TARGET_FILE_MB} MB -> {files} files")
print(f"average file size = {avg_file_mb:.1f} MB")
print("small-file-bound:", small_file_bound)

assert files == 760
assert round(avg_file_mb, 1) == 256.0
assert small_file_bound is False


### Try it 4. Size a resource pool

**Task.** Ten workers each give Impala 200 GB, and the pool may use 80 percent of that. A typical dashboard query needs 20 GB and a typical ETL query needs 60 GB. Compute the pool memory and the maximum concurrency for each query shape.

**Hint.** `pool_gb = nodes * mem_per_node_gb * 0.8`; concurrency is integer division by the per-query memory.

**Expected result.** Pool `1600.0 GB`, `80` dashboard queries or `26` ETL queries.

In [ ]:
nodes = 10
mem_per_node_gb = 200
POOL_SHARE = 0.8

pool_gb = nodes * mem_per_node_gb * POOL_SHARE
dashboard_concurrency = int(pool_gb // 20)
etl_concurrency = int(pool_gb // 60)

print(f"pool memory = {pool_gb:,.1f} GB")
print(f"dashboard (20 GB) concurrency = {dashboard_concurrency}")
print(f"ETL (60 GB) concurrency       = {etl_concurrency}")

assert round(pool_gb, 1) == 1600.0
assert dashboard_concurrency == 80
assert etl_concurrency == 26


### Try it 5. Compute spill risk for an operator

**Task.** A hash join build side is 40 GB, the aggregate needs 12 GB, and the sort needs 18 GB. The per-query limit is 32 GB and the scratch limit is 100 GB. Identify which operators must spill, total the estimated scratch space, and decide whether the scratch limit is enough.

**Hint.** An operator spills when its memory exceeds `MEM_LIMIT_GB`; the scratch it needs is the excess over the limit.

**Expected result.** Only `hash join build` spills, needing `8 GB` of scratch, which fits under the `100 GB` limit.

In [ ]:
MEM_LIMIT_GB = 32
SCRATCH_LIMIT_GB = 100
operators = {"hash join build": 40, "aggregate": 12, "sort": 18}

spilling = {}
for name, mem_gb in operators.items():
    if mem_gb > MEM_LIMIT_GB:
        spilling[name] = mem_gb - MEM_LIMIT_GB

scratch_gb = sum(spilling.values())
print("spilling operators:", spilling)
print(f"scratch needed = {scratch_gb} GB, limit = {SCRATCH_LIMIT_GB} GB")

assert list(spilling) == ["hash join build"]
assert scratch_gb == 8
assert scratch_gb <= SCRATCH_LIMIT_GB


### Try it 6. Choose a partition column

**Task.** Query logs show how often each column appears in a filter. A good partition column is filtered often and has a moderate number of distinct values. Choose the column with the most filters among the candidates whose cardinality is at most 10000.

**Hint.** Filter the candidates first, then take the one with the highest filter count.

**Expected result.** Partition by `order_date` (120 filters, 3650 distinct values).

In [ ]:
filter_counts = {"order_date": 120, "customer_id": 90, "region": 40, "status": 25}
cardinality = {"order_date": 3650, "customer_id": 2_000_000, "region": 12, "status": 6}
MAX_CARDINALITY = 10_000

candidates = {c: filter_counts[c] for c in filter_counts if cardinality[c] <= MAX_CARDINALITY}
chosen = max(candidates, key=candidates.get)
print("eligible:", sorted(candidates))
print(f"chosen partition column = {chosen} "
      f"({filter_counts[chosen]} filters, {cardinality[chosen]} distinct values)")

assert chosen == "order_date"
assert cardinality[chosen] <= MAX_CARDINALITY


### Try it 7. Estimate statistics-collection cost

**Task.** A table has 4 billion rows at 120 bytes per row and spans 1095 daily partitions. The cluster collects statistics at 4 million rows per second. Compute the full-collection time in minutes and the per-partition time in seconds.

**Hint.** `seconds = rows / rows_per_second`, then divide by the partition count for the incremental case.

**Expected result.** Full collection takes about `16.7` minutes; one partition takes about `0.9` seconds.

In [ ]:
rows = 4_000_000_000
avg_row_bytes = 120
rows_per_second = 4_000_000
partitions = 1095

full_seconds = rows / rows_per_second
partition_seconds = full_seconds / partitions

print(f"table size = {rows * avg_row_bytes / 1024 ** 3:,.0f} GB")
print(f"full COMPUTE STATS   = {full_seconds / 60:.1f} minutes")
print(f"one partition        = {partition_seconds:.2f} seconds")

assert round(full_seconds / 60, 1) == 16.7
assert round(partition_seconds, 2) == 0.91


### Try it 8. Decide whether a query is shuffle-bound

**Task.** A query scans 1.9 GB but produces an exchange carrying 1.2 billion rows at 40 bytes per row. Compare the exchange volume with the scan volume and decide whether the query is scan-bound or shuffle-bound.

**Hint.** `exchange_gb = rows * bytes_per_row / 1024 ** 3`; compare it with the scan size.

**Expected result.** The exchange carries `44.7 GB` versus a `1.9 GB` scan, so the query is `shuffle-bound`.

In [ ]:
scan_gb = 1.9
exchange_rows = 1_200_000_000
exchange_bytes_per_row = 40

exchange_gb = exchange_rows * exchange_bytes_per_row / 1024 ** 3
verdict = "shuffle-bound" if exchange_gb > scan_gb else "scan-bound"

print(f"scan     = {scan_gb} GB")
print(f"exchange = {exchange_gb:.1f} GB")
print("verdict:", verdict)

assert round(exchange_gb, 1) == 44.7
assert verdict == "shuffle-bound"
